# 05 — Ground an answer in company knowledge
Context is what goes into this run. Knowledge is the document collection
searched for relevant passages. Source IDs make the answer auditable.



**Basic lesson 5 of 13.**


## Problem

A fluent answer may omit the company documents or invent vendor claims.


## Concept

CrewAI Knowledge retrieves relevant document chunks using a configured embedder.

Knowledge is retrieved source material, not conversation history. Chunking and embeddings select relevant passages; source IDs let a reviewer connect an output claim back to an input document.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents   ◀ YOU ARE HERE
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
company files ──▶ Knowledge sources ──▶ chunks + embedder
                                               │ retrieval
user question ──▶ Crew / Researcher ◀──────────┘
                         └──▶ answer + source IDs + unknowns
```

sample documents → chunks/embeddings → Knowledge retrieval → agent answer. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Inspect the document search and explicit embedding configuration

Inspect the document search and explicit embedding configuration.


In [ ]:
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

from workshop.common import DATA, embedder_config, model
from workshop.tools import DOCUMENTS, search_documents

print("Transparent search before RAG:", search_documents("private deployment security")[:1])
sources = [
    StringKnowledgeSource(content=f"Source ID: {name}\n{(DATA / name).read_text(encoding='utf-8')}")
    for name in DOCUMENTS
]
embedder = embedder_config()
print("CrewAI Knowledge sources:", len(sources))
print("Embedding provider:", embedder["provider"])



## CrewAI Knowledge retrieval
The embedder is explicit. Changing the chat model alone does not configure
Knowledge or Memory embeddings.



## Step 2 — Attach Knowledge sources to the Crew and demand source IDs and unknowns in the answer

Attach Knowledge sources to the Crew and demand source IDs and unknowns in the answer.


In [ ]:
researcher = Agent(
    role="Company Knowledge Researcher",
    goal="Answer only from named sample company sources",
    backstory="You cite the source ID for every factual claim.",
    llm=model(),
    max_iter=3,
    allow_delegation=False,
)
task = Task(
    description="What is known and unknown about Meridian AI's security? "
    "Cite source IDs. Say when evidence is missing.",
    expected_output="Short answer with source IDs and unknowns.",
    agent=researcher,
)
crew = Crew(
    agents=[researcher], tasks=[task], process=Process.sequential,
    knowledge_sources=sources, embedder=embedder,
)
print(crew.kickoff().raw)



## Bonus scenario — matching a candidate
Knowledge works on any documents (inspired by CrewAI's *Recruitment* example).
Here three short fictional profiles replace the company files.
This run takes about as long as the one above; skip it if time is short.



In [ ]:
profiles = [
    StringKnowledgeSource(content="Candidate Asha: 6 years Python, built two CrewAI agents, no cloud experience."),
    StringKnowledgeSource(content="Candidate Ben: 2 years Java, strong on Kubernetes and deployment."),
    StringKnowledgeSource(content="Candidate Chen: 4 years data science, wrote evaluation suites for LLM apps."),
]
screener = Agent(
    role="Talent Screener",
    goal="Recommend the best-matching candidate using only the profiles",
    backstory="You name the candidate and the evidence.",
    llm=model(),
    max_iter=3,
    allow_delegation=False,
)
match = Task(
    description="Who best fits an 'agent engineer who writes Python' role? Name one candidate and why.",
    expected_output="One candidate name and the profile fact that supports it.",
    agent=screener,
)
print(Crew(
    agents=[screener], tasks=[match], process=Process.sequential,
    knowledge_sources=profiles, embedder=embedder,
).kickoff().raw)


## Failure mode to know about

Retrieval can return an irrelevant chunk or no chunk. Source IDs in prose are not automatic proof: verify them against the actual documents and explicitly report missing evidence.


## Production implication

Define document ownership, retention, and re-indexing rules. Evaluate retrieval relevance and source attribution separately from answer fluency.


## Your turn

Ask about deletion guarantees and inspect whether the answer admits the gap.

**Checkpoint:** Answer claims name source IDs and identify unverified facts.
